# T-05 - Bonus Evaluation

**Three lightweight bonus additions**

1. a cleaner demo package with prepared inputs, outputs, and a README;
2. one additional result slice based on low / medium / high retrieval similarity;
3. a controlled comparison between the original prompt and one stricter abstention prompt.

This notebook keeps the original Phase 1 and Phase 2 submissions unchanged. It reuses the
exact Phase 1 sample, corpus, FAISS index, embedding model, and generator. There is no training,
fine-tuning, threshold tuning, or new model.

**Before running:** In Colab, choose `Runtime > Change runtime type > T4 GPU`.
Then run every cell from top to bottom and upload `T05_Phase1_Outputs.zip` when requested.

## 1. Install the same lightweight dependencies

These versions match the original project. The comparison changes only the prompt text.

In [ ]:
%pip -q install \
    sentence-transformers==5.7.0 \
    faiss-cpu==1.15.0 \
    transformers==5.15.0 \
    sentencepiece==0.2.2 \
    matplotlib==3.10.8

## 2. Imports, fixed settings, and output folder

In [ ]:
from pathlib import Path
from collections import Counter
import io
import json
import random
import re
import shutil
import string
import zipfile

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SEED = 42
PHASE1_DIR = Path("/content/T05_Phase1_Outputs")
OUTPUT_DIR = Path("/content/T05_Bonus_Outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("Warning: the notebook works on CPU, but T4 GPU is much faster.")

## 3. Upload and validate the Phase 1 artifacts

Only `T05_Phase1_Outputs.zip` is needed. The notebook reconstructs the unchanged retrieval
evidence and reruns both prompts under identical conditions.

In [ ]:
if not (PHASE1_DIR / "phase1_config.json").exists():
    print("Upload T05_Phase1_Outputs.zip")
    try:
        from google.colab import files
        uploaded = files.upload()
    except ImportError as exc:
        raise FileNotFoundError(
            "Place T05_Phase1_Outputs.zip in /content before running this cell."
        ) from exc

    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise ValueError("No ZIP file was uploaded.")

    preferred = [name for name in zip_names if "phase1" in name.lower()]
    selected_zip = preferred[0] if preferred else zip_names[0]
    PHASE1_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(uploaded[selected_zip])) as archive:
        archive.extractall(PHASE1_DIR)

required_files = [
    "phase1_config.json",
    "phase1_questions.csv",
    "phase1_corpus.csv",
    "corpus_embeddings.npy",
    "corpus.faiss",
]
missing = [name for name in required_files if not (PHASE1_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing Phase 1 artifacts: {missing}")

print("Phase 1 artifacts are ready:", PHASE1_DIR)

## 4. Restore the exact baseline assets

The saved question sample, corpus order, FAISS index, seed, model names, and Top-K value all
come from Phase 1. The original project remains the baseline; this notebook is an extension.

In [ ]:
with (PHASE1_DIR / "phase1_config.json").open(encoding="utf-8") as file:
    config = json.load(file)

questions = pd.read_csv(PHASE1_DIR / "phase1_questions.csv")
corpus = pd.read_csv(PHASE1_DIR / "phase1_corpus.csv")
questions["gold_answers"] = questions["gold_answers_json"].apply(json.loads)

if questions["is_answerable"].dtype != bool:
    questions["is_answerable"] = (
        questions["is_answerable"].astype(str).str.lower().eq("true")
    )

TOP_K = int(config["retrieval_top_k"])
EMBEDDING_MODEL = config["embedding_model"]
GENERATOR_MODEL = config["generator_model"]

embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
index = faiss.read_index(str(PHASE1_DIR / "corpus.faiss"))
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL)
generator = AutoModelForSeq2SeqLM.from_pretrained(GENERATOR_MODEL).to(DEVICE)
generator.eval()

assert len(questions) == 120, "The bonus notebook expects the fixed 120-question sample."
assert index.ntotal == len(corpus), "FAISS index and corpus row count do not match."

print("Questions:", len(questions))
print("Answerable / unanswerable:", questions["is_answerable"].value_counts().to_dict())
print("Corpus passages:", len(corpus))
print("Embedding model:", EMBEDDING_MODEL)
print("Generator model:", GENERATOR_MODEL)

## 5. Restore the same rank-1 evidence for every question

Retrieval is performed once. Both prompts receive the exact same retrieved passage, so prompt
text is the only experimental variable.

In [ ]:
query_embeddings = embedder.encode(
    questions["question"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
).astype("float32")

similarities, positions = index.search(query_embeddings, TOP_K)

retrieval_rows = []
for row_number, question_row in questions.iterrows():
    selected = corpus.iloc[positions[row_number]].reset_index(drop=True)
    retrieved_ids = selected["context_id"].tolist()
    retrieval_rows.append({
        "id": question_row["id"],
        "top1_context_id": retrieved_ids[0],
        "top1_title": selected.iloc[0]["title"],
        "top1_context": selected.iloc[0]["context"],
        "top1_similarity": float(similarities[row_number][0]),
        "retrieval_hit_top1": question_row["gold_context_id"] == retrieved_ids[0],
        "retrieval_hit_top3": question_row["gold_context_id"] in retrieved_ids,
    })

results = questions.merge(pd.DataFrame(retrieval_rows), on="id", validate="one_to_one")

answerable = results[results["is_answerable"]]
print("Answerable hit@1:", f"{100 * answerable['retrieval_hit_top1'].mean():.2f}%")
print("Answerable hit@3:", f"{100 * answerable['retrieval_hit_top3'].mean():.2f}%")
display(results[["question", "top1_title", "top1_similarity"]].head())

## 6. Define the two prompts

- **Baseline prompt:** copied exactly from the original project.
- **Strict prompt:** asks for explicit evidence, forbids outside knowledge and inference, and
  strengthens the `NO_ANSWER` instruction.

The model, evidence, decoding, maximum input length, and output length are identical.

In [ ]:
def build_baseline_prompt(question, passage):
    return f"""Answer the question using ONLY the passage below.
If the passage does not contain enough evidence, output exactly NO_ANSWER.
Keep the answer short and copy the answer span when possible.

Passage:
{passage}

Question: {question}
Answer:"""


def build_strict_prompt(question, passage):
    return f"""You are a conservative extractive question-answering system.
Use only information that is explicitly stated in the passage.
Do not use outside knowledge. Do not guess or infer missing facts.
If the passage does not explicitly contain the answer, output exactly NO_ANSWER.
If the answer is present, output only the shortest supported answer span.

Passage:
{passage}

Question: {question}
Answer:"""


def standardize_answer(text):
    cleaned = text.strip()
    no_answer_forms = {"no_answer", "no answer", "no-answer", "none"}
    return "NO_ANSWER" if cleaned.lower() in no_answer_forms else cleaned


def generate_in_batches(prompts, batch_size=8):
    predictions = []
    for start in range(0, len(prompts), batch_size):
        batch = prompts[start:start + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(DEVICE)
        with torch.inference_mode():
            output_ids = generator.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=False,
                num_beams=1,
            )
        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        predictions.extend(standardize_answer(text) for text in decoded)
    return predictions


baseline_prompts = [
    build_baseline_prompt(row.question, row.top1_context)
    for row in results.itertuples(index=False)
]
strict_prompts = [
    build_strict_prompt(row.question, row.top1_context)
    for row in results.itertuples(index=False)
]

print("Running baseline prompt on 120 fixed examples...")
results["baseline_prediction"] = generate_in_batches(baseline_prompts)
print("Running strict prompt on the same 120 examples...")
results["strict_prediction"] = generate_in_batches(strict_prompts)
print("Both prompt runs are complete.")

## 7. Compute the same SQuAD Exact Match and Token F1 metrics

Unanswerable examples have an empty gold answer. A prediction receives full credit on those
examples only when it is standardized to `NO_ANSWER` and then mapped to an empty metric answer.

In [ ]:
def normalize_answer(text):
    def remove_articles(value):
        return re.sub(r"\b(a|an|the)\b", " ", value)

    def remove_punctuation(value):
        return "".join(
            character for character in value if character not in string.punctuation
        )

    return " ".join(remove_articles(remove_punctuation(text.lower())).split())


def exact_match_score(prediction, gold_answers):
    golds = gold_answers if gold_answers else [""]
    return int(any(
        normalize_answer(prediction) == normalize_answer(gold)
        for gold in golds
    ))


def token_f1_score(prediction, gold_answers):
    golds = gold_answers if gold_answers else [""]
    prediction_tokens = normalize_answer(prediction).split()
    scores = []

    for gold in golds:
        gold_tokens = normalize_answer(gold).split()
        common = Counter(prediction_tokens) & Counter(gold_tokens)
        shared = sum(common.values())

        if not prediction_tokens and not gold_tokens:
            scores.append(1.0)
        elif not prediction_tokens or not gold_tokens or shared == 0:
            scores.append(0.0)
        else:
            precision = shared / len(prediction_tokens)
            recall = shared / len(gold_tokens)
            scores.append(2 * precision * recall / (precision + recall))

    return max(scores)


for prompt_name in ["baseline", "strict"]:
    prediction_column = f"{prompt_name}_prediction"
    abstained_column = f"{prompt_name}_abstained"
    metric_prediction_column = f"{prompt_name}_prediction_for_metric"

    results[abstained_column] = results[prediction_column].eq("NO_ANSWER")
    results[metric_prediction_column] = results[prediction_column].where(
        ~results[abstained_column], ""
    )
    results[f"{prompt_name}_exact_match"] = results.apply(
        lambda row: exact_match_score(
            row[metric_prediction_column], row["gold_answers"]
        ),
        axis=1,
    )
    results[f"{prompt_name}_token_f1"] = results.apply(
        lambda row: token_f1_score(
            row[metric_prediction_column], row["gold_answers"]
        ),
        axis=1,
    )

print("Metrics computed for both prompts.")

## 8. Bonus A - Controlled prompt comparison

This is a one-variable comparison: only the prompt changes. The table reports overall quality,
answerable quality, and unanswerable abstention separately so that a prompt is not called better
merely because it abstains more often.

In [ ]:
def safe_percent(series):
    return round(100 * series.mean(), 2) if len(series) else None


def summarize_prompt(frame, prompt_name, display_name):
    answerable_frame = frame[frame["is_answerable"]]
    unanswerable_frame = frame[~frame["is_answerable"]]
    return {
        "prompt": prompt_name,
        "display_name": display_name,
        "examples": int(len(frame)),
        "exact_match_percent": safe_percent(frame[f"{prompt_name}_exact_match"]),
        "token_f1_percent": safe_percent(frame[f"{prompt_name}_token_f1"]),
        "answerable_exact_match_percent": safe_percent(
            answerable_frame[f"{prompt_name}_exact_match"]
        ),
        "answerable_token_f1_percent": safe_percent(
            answerable_frame[f"{prompt_name}_token_f1"]
        ),
        "unanswerable_abstention_accuracy_percent": safe_percent(
            unanswerable_frame[f"{prompt_name}_abstained"]
        ),
        "overall_abstention_rate_percent": safe_percent(
            frame[f"{prompt_name}_abstained"]
        ),
    }


prompt_comparison = pd.DataFrame([
    summarize_prompt(results, "baseline", "Original prompt"),
    summarize_prompt(results, "strict", "Strict abstention prompt"),
])

metric_columns = [
    "exact_match_percent",
    "token_f1_percent",
    "answerable_exact_match_percent",
    "answerable_token_f1_percent",
    "unanswerable_abstention_accuracy_percent",
    "overall_abstention_rate_percent",
]

delta = {"prompt": "strict_minus_baseline", "display_name": "Difference"}
for column in metric_columns:
    delta[column] = round(
        prompt_comparison.loc[1, column] - prompt_comparison.loc[0, column], 2
    )
delta["examples"] = int(len(results))

prompt_comparison_with_delta = pd.concat(
    [prompt_comparison, pd.DataFrame([delta])], ignore_index=True
)
display(prompt_comparison_with_delta)

results["prompt_effect"] = np.select(
    [
        results["strict_exact_match"] > results["baseline_exact_match"],
        results["strict_exact_match"] < results["baseline_exact_match"],
        results["strict_abstained"] != results["baseline_abstained"],
        results["strict_prediction"] != results["baseline_prediction"],
    ],
    [
        "strict_corrected_exact_match",
        "strict_harmed_exact_match",
        "abstention_changed_same_exact_match",
        "wording_changed_same_exact_match",
    ],
    default="unchanged",
)

prompt_effect_counts = (
    results["prompt_effect"]
    .value_counts()
    .rename_axis("prompt_effect")
    .reset_index(name="count")
)

change_categories = [
    "strict_corrected_exact_match",
    "strict_harmed_exact_match",
    "abstention_changed_same_exact_match",
    "wording_changed_same_exact_match",
]
prompt_change_examples = pd.concat(
    [results[results["prompt_effect"] == category].head(3) for category in change_categories],
    ignore_index=True,
)

print("How often did the strict prompt change behavior?")
display(prompt_effect_counts)
print("Representative changed outputs (up to three per change type)")
display(prompt_change_examples[[
    "prompt_effect",
    "question",
    "baseline_prediction",
    "strict_prediction",
    "is_answerable",
    "top1_similarity",
]])

baseline_em = prompt_comparison.loc[0, "exact_match_percent"]
baseline_f1 = prompt_comparison.loc[0, "token_f1_percent"]
expected_baselines = {
    "google/flan-t5-small": (43.33, 50.66),
    "google/flan-t5-base": (50.83, 59.09),
}
expected_em, expected_f1 = expected_baselines.get(
    GENERATOR_MODEL, (baseline_em, baseline_f1)
)
if abs(baseline_em - expected_em) > 0.05 or abs(baseline_f1 - expected_f1) > 0.05:
    print(
        "Warning: the rerun baseline differs from the earlier report. "
        "Keep this run's results together and report the model/library environment."
    )
else:
    print(f"Baseline reproduction check passed: {expected_em:.2f}% EM and {expected_f1:.2f}% Token F1.")

comparison_plot = prompt_comparison.set_index("display_name")[[
    "exact_match_percent",
    "token_f1_percent",
    "unanswerable_abstention_accuracy_percent",
]]
ax = comparison_plot.plot(
    kind="bar",
    figsize=(9, 5),
    color=["#4DA3FF", "#08B8B3", "#E24A4A"],
)
ax.set_title("Controlled prompt comparison")
ax.set_xlabel("")
ax.set_ylabel("Percent")
ax.set_ylim(0, 100)
ax.tick_params(axis="x", rotation=0)
ax.legend(["Exact Match", "Token F1", "Unanswerable abstention"], frameon=False)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "prompt_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

## 9. Bonus B - Similarity-based result slice

The 120 examples are divided into three equal-size descriptive groups using the rank of their
Top-1 cosine similarity: low, medium, and high. This avoids choosing a favorable threshold.
Similarity is analyzed as an association only; it is **not** treated as a calibrated confidence
score or an abstention threshold.

In [ ]:
similarity_labels = ["low", "medium", "high"]
results["similarity_group"] = pd.qcut(
    results["top1_similarity"].rank(method="first"),
    q=3,
    labels=similarity_labels,
)


def summarize_similarity_group(frame, prompt_name, group_name):
    answerable_frame = frame[frame["is_answerable"]]
    unanswerable_frame = frame[~frame["is_answerable"]]
    return {
        "prompt": prompt_name,
        "similarity_group": str(group_name),
        "examples": int(len(frame)),
        "similarity_min": round(float(frame["top1_similarity"].min()), 4),
        "similarity_max": round(float(frame["top1_similarity"].max()), 4),
        "similarity_mean": round(float(frame["top1_similarity"].mean()), 4),
        "answerable_examples": int(len(answerable_frame)),
        "unanswerable_examples": int(len(unanswerable_frame)),
        "exact_match_percent": safe_percent(frame[f"{prompt_name}_exact_match"]),
        "token_f1_percent": safe_percent(frame[f"{prompt_name}_token_f1"]),
        "answerable_exact_match_percent": safe_percent(
            answerable_frame[f"{prompt_name}_exact_match"]
        ),
        "answerable_token_f1_percent": safe_percent(
            answerable_frame[f"{prompt_name}_token_f1"]
        ),
        "unanswerable_abstention_accuracy_percent": safe_percent(
            unanswerable_frame[f"{prompt_name}_abstained"]
        ),
        "overall_abstention_rate_percent": safe_percent(
            frame[f"{prompt_name}_abstained"]
        ),
    }


similarity_rows = []
for prompt_name in ["baseline", "strict"]:
    for group_name in similarity_labels:
        group_frame = results[results["similarity_group"] == group_name]
        similarity_rows.append(
            summarize_similarity_group(group_frame, prompt_name, group_name)
        )

similarity_breakdown = pd.DataFrame(similarity_rows)
display(similarity_breakdown)

baseline_similarity = (
    similarity_breakdown[similarity_breakdown["prompt"] == "baseline"]
    .set_index("similarity_group")
    .reindex(similarity_labels)
)

ax = baseline_similarity[["exact_match_percent", "token_f1_percent"]].plot(
    kind="bar",
    figsize=(9, 5),
    color=["#4DA3FF", "#08B8B3"],
)
ax.set_title("Original baseline quality by Top-1 similarity group")
ax.set_xlabel("Top-1 similarity group (40 examples each)")
ax.set_ylabel("Percent")
ax.set_ylim(0, 100)
ax.tick_params(axis="x", rotation=0)
ax.legend(["Exact Match", "Token F1"], frameon=False)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "similarity_breakdown.png", dpi=180, bbox_inches="tight")
plt.show()

## 10. Bonus C - Cleaner demo package

The demo contains six explainable cases:

- high- and low-similarity answerable benchmark questions;
- high- and low-similarity unanswerable benchmark questions;
- two realistic questions that are outside the corpus.

Both prompts are shown for every case. The function below can also be called interactively with
a new question after the notebook finishes.

In [ ]:
def retrieve_one(question):
    query = embedder.encode([question], normalize_embeddings=True).astype("float32")
    scores, found_positions = index.search(query, 1)
    passage = corpus.iloc[int(found_positions[0][0])]
    return {
        "source_title": passage["title"],
        "source_context_id": passage["context_id"],
        "source_context": passage["context"],
        "retrieval_similarity": round(float(scores[0][0]), 4),
    }


def answer_one_question(question):
    retrieved = retrieve_one(question)
    passage = retrieved["source_context"]
    baseline_answer = generate_in_batches(
        [build_baseline_prompt(question, passage)], batch_size=1
    )[0]
    strict_answer = generate_in_batches(
        [build_strict_prompt(question, passage)], batch_size=1
    )[0]
    return {
        "question": question,
        "baseline_answer": baseline_answer,
        "baseline_abstained": baseline_answer == "NO_ANSWER",
        "strict_answer": strict_answer,
        "strict_abstained": strict_answer == "NO_ANSWER",
        **retrieved,
    }


def selected_benchmark_case(frame, case_id, case_type, expected_behavior):
    row = frame.iloc[0]
    return {
        "case_id": case_id,
        "case_type": case_type,
        "question": row["question"],
        "gold_answers_json": row["gold_answers_json"],
        "expected_behavior": expected_behavior,
    }


answerable_sorted = results[results["is_answerable"]].sort_values("top1_similarity")
unanswerable_sorted = results[~results["is_answerable"]].sort_values("top1_similarity")

demo_inputs = pd.DataFrame([
    selected_benchmark_case(
        answerable_sorted.tail(1),
        "A-HIGH",
        "answerable_high_similarity",
        "answer according to the SQuAD gold answer",
    ),
    selected_benchmark_case(
        answerable_sorted.head(1),
        "A-LOW",
        "answerable_low_similarity",
        "answer according to the SQuAD gold answer",
    ),
    selected_benchmark_case(
        unanswerable_sorted.tail(1),
        "U-HIGH",
        "unanswerable_high_similarity",
        "abstain according to the SQuAD label",
    ),
    selected_benchmark_case(
        unanswerable_sorted.head(1),
        "U-LOW",
        "unanswerable_low_similarity",
        "abstain according to the SQuAD label",
    ),
    {
        "case_id": "CUSTOM-1",
        "case_type": "outside_corpus_private_information",
        "question": "What is the instructor's private email address?",
        "gold_answers_json": "[]",
        "expected_behavior": "abstain because the private information is outside the corpus",
    },
    {
        "case_id": "CUSTOM-2",
        "case_type": "outside_corpus_course_information",
        "question": "What is the deadline of our next artificial intelligence assignment?",
        "gold_answers_json": "[]",
        "expected_behavior": "abstain because the course deadline is outside the corpus",
    },
])

demo_rows = []
for case in demo_inputs.itertuples(index=False):
    output = answer_one_question(case.question)
    demo_rows.append({
        "case_id": case.case_id,
        "case_type": case.case_type,
        "question": case.question,
        "gold_answers_json": case.gold_answers_json,
        "expected_behavior": case.expected_behavior,
        **output,
    })

demo_outputs = pd.DataFrame(demo_rows)
display(demo_outputs[[
    "case_id",
    "case_type",
    "question",
    "baseline_answer",
    "strict_answer",
    "retrieval_similarity",
    "source_title",
]])

print("Interactive demo is ready. Example:")
print("answer_one_question(\"What is the main judicial body of the EU?\")")

## 11. Save all bonus artifacts

The final ZIP is the only output you need to send back. It contains raw predictions, tables,
charts, demo inputs/outputs, prompt text, a demo README, and a manifest.

In [ ]:
prediction_columns = [
    "id",
    "title",
    "question",
    "gold_answers_json",
    "is_answerable",
    "gold_context_id",
    "top1_context_id",
    "top1_title",
    "top1_context",
    "top1_similarity",
    "similarity_group",
    "retrieval_hit_top1",
    "retrieval_hit_top3",
    "baseline_prediction",
    "baseline_abstained",
    "baseline_exact_match",
    "baseline_token_f1",
    "strict_prediction",
    "strict_abstained",
    "strict_exact_match",
    "strict_token_f1",
    "prompt_effect",
]

results[prediction_columns].to_csv(
    OUTPUT_DIR / "prompt_comparison_predictions.csv", index=False
)
prompt_comparison_with_delta.to_csv(
    OUTPUT_DIR / "prompt_comparison_metrics.csv", index=False
)
prompt_effect_counts.to_csv(
    OUTPUT_DIR / "prompt_effect_counts.csv", index=False
)
prompt_change_examples_export = pd.concat(
    [results[results["prompt_effect"] == category].head(3) for category in change_categories],
    ignore_index=True,
)
prompt_change_examples_export[prediction_columns].to_csv(
    OUTPUT_DIR / "prompt_change_examples.csv", index=False
)
similarity_breakdown.to_csv(
    OUTPUT_DIR / "similarity_breakdown.csv", index=False
)
demo_inputs.to_csv(OUTPUT_DIR / "demo_inputs.csv", index=False)
demo_outputs.to_csv(OUTPUT_DIR / "demo_outputs.csv", index=False)

bonus_config = {
    "seed": SEED,
    "dataset": config["dataset_id"],
    "dataset_split": config["dataset_split"],
    "examples": int(len(results)),
    "embedding_model": EMBEDDING_MODEL,
    "generator_model": GENERATOR_MODEL,
    "retrieval_top_k": TOP_K,
    "generator_passages_used": 1,
    "decoding": {
        "do_sample": False,
        "num_beams": 1,
        "max_new_tokens": 32,
        "max_input_tokens": 512,
    },
    "baseline_prompt": build_baseline_prompt("{question}", "{passage}"),
    "strict_prompt": build_strict_prompt("{question}", "{passage}"),
    "similarity_slice": {
        "method": "three equal-size groups using ranked Top-1 cosine similarity",
        "labels": similarity_labels,
        "interpretation_limit": (
            "Descriptive association only; not a calibrated confidence score or threshold."
        ),
    },
}
with (OUTPUT_DIR / "bonus_config.json").open("w", encoding="utf-8") as file:
    json.dump(bonus_config, file, indent=2, ensure_ascii=False)

baseline_row = prompt_comparison.iloc[0]
strict_row = prompt_comparison.iloc[1]
bonus_summary = {
    "baseline_reproduction": {
        "exact_match_percent": float(baseline_row["exact_match_percent"]),
        "token_f1_percent": float(baseline_row["token_f1_percent"]),
    },
    "strict_minus_baseline_percentage_points": {
        column: float(delta[column]) for column in metric_columns
    },
    "interpretation_rules": [
        "Do not call the strict prompt better based only on a higher abstention rate.",
        "Inspect answerable quality and unanswerable abstention together.",
        "Similarity groups are descriptive and are not an abstention threshold.",
        "Custom demo questions have expected behavior but are not included in SQuAD metrics.",
    ],
}
with (OUTPUT_DIR / "bonus_summary.json").open("w", encoding="utf-8") as file:
    json.dump(bonus_summary, file, indent=2, ensure_ascii=False)

demo_readme = """# T-05 Cleaner Demo Package

## Purpose

This package demonstrates the unchanged lightweight RAG pipeline and compares the original
prompt with one stricter abstention prompt.

## Included demo cases

- two answerable SQuAD v2 questions at high and low Top-1 similarity;
- two unanswerable SQuAD v2 questions at high and low Top-1 similarity;
- two realistic out-of-corpus questions that should trigger abstention.

## Files

- `demo_inputs.csv`: prepared questions and expected behavior;
- `demo_outputs.csv`: answers from both prompts, abstention flags, source, and similarity;
- `prompt_comparison_predictions.csv`: all 120 controlled comparison rows;
- `prompt_comparison_metrics.csv`: overall comparison and percentage-point difference;
- `prompt_effect_counts.csv`: counts of corrected, harmed, changed, and unchanged outputs;
- `prompt_change_examples.csv`: representative examples where the strict prompt changed behavior;
- `similarity_breakdown.csv`: low / medium / high similarity analysis;
- `bonus_config.json`: exact prompts, models, and decoding settings;
- `bonus_summary.json`: compact numeric summary and interpretation rules;
- `prompt_comparison.png` and `similarity_breakdown.png`: report-ready figures.

## Interactive use

Run `T05_Bonus_Evaluation.ipynb` from top to bottom in Colab and call:

```python
answer_one_question("your question")
```

The returned dictionary includes answers from both prompts, abstention flags, the retrieved
source title, context ID, full source context, and retrieval similarity.

## Important limitation

Top-1 similarity is not a calibrated confidence score. The three similarity groups are only a
descriptive result slice and must not be presented as a tuned abstention threshold.
"""
(OUTPUT_DIR / "README_DEMO.md").write_text(demo_readme, encoding="utf-8")

manifest = {
    "files": sorted(path.name for path in OUTPUT_DIR.iterdir()),
    "bonus_items": [
        "cleaner demo package",
        "Top-1 similarity result slice",
        "controlled original-vs-strict prompt comparison",
    ],
    "note": "Send the complete T05_Bonus_Outputs.zip for report and presentation updates.",
}
with (OUTPUT_DIR / "bonus_manifest.json").open("w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2, ensure_ascii=False)

archive = shutil.make_archive(
    "/content/T05_Bonus_Outputs", "zip", root_dir=OUTPUT_DIR
)
print("Created:", archive)
print("Files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Download manually from:", archive)

## Finished

Send back only `T05_Bonus_Outputs.zip`. The report and presentation should be updated only after
these real execution results are available. Do not claim bonus improvement before inspecting the
prompt comparison table and its effect on both answerable quality and unanswerable abstention.